# 价值因子 (HML) 完整教程：从 B/M 分组到多空组合检验

## 📚 教学目标
1. 理解**价值因子**的定义：按 B/M（账面市值比）分组，高 B/M − 低 B/M = HML
2. 掌握 **B/M 的计算方法**和数据处理（滞后处理避免前视偏差）
3. 在**微型数据集**（10 只股票 × 1 月）上手算 B/M 分组和 HML Spread
4. 扩展到 **300 只股票 × 60 月**，完成 T 检验和单调性检验
5. 理解 **A 股价值效应**的时变特征

**参考书**：《因子投资：方法与实践》（石川）第 3.4 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 什么是价值因子？为什么低估值股票收益更高？

### 🎯 一个直觉问题

假设你面前有两家公司：A 公司股价 10 元，每股净资产 8 元（B/M=0.8）；B 公司股价 100 元，每股净资产 10 元（B/M=0.1）。你更愿意买哪家？

价值投资的理念是买"便宜"的股票。**价值因子 (HML, High-Minus-Low)** 就是检验这个理念：做高 B/M（便宜）股票、做空低 B/M（贵）股票，能否获得显著的超额收益？

### 📖 书中的定义

> 价值因子的含义简单而明确：相比估值较高的股票，那些估值较低的股票有着更高的预期收益率。

> Stattman（1980）是最早的相关研究之一，它发现 BM 较高的公司，股票预期收益也显著更高。

### 📐 价值因子的理论基础

| 理论来源 | 核心逻辑 |
|---------|----------|
| **财务困境风险** | 高 B/M 企业可能面临更大财务困境 → 风险补偿 |
| **经营杠杆** | 不景气时难以削减在位资产 → 高 B/M 企业风险更大 |
| **投资者外推偏差** | 投资者将过去表现简单外推 → 对低估值企业过度悲观 |
| **无形信息忽视** | 投资者对有形信息关注不足 → B/M 能预测无形收益 |

### 📐 常用价值指标

| 指标 | 公式 | 特点 |
|------|------|------|
| **B/M** | 每股净资产 / 股价 | 最经典，FF3/FF5 采用 |
| **E/P** | 每股盈利 / 股价 | 盈利市值比 |
| **CF/P** | 每股现金流 / 股价 | 现金流市值比 |
| **D/P** | 每股股利 / 股价 | 股息率 |

### 📐 HML 因子的定义

$$\text{HML} = R_{\text{High BM}} - R_{\text{Low BM}}$$

其中：
- $R_{\text{High BM}}$ = 高 B/M 组（价值股）的组合收益率
- $R_{\text{Low BM}}$ = 低 B/M 组（成长股）的组合收益率
- HML > 0 表示价值股收益高于成长股

In [ ]:
import sys, os
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")
try:
    import matplotlib
    print(f"matplotlib: {matplotlib.__version__}")
except ImportError:
    print("❌ matplotlib 未安装! 请运行: !pip install matplotlib seaborn statsmodels scipy")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 微型数据集：10 只股票的手算演示

### 🎯 场景

假设市场上只有 **10 只股票**，我们用 B/M（账面市值比）作为价值指标，检验高 B/M 股票是否获得更高收益。

### 📐 数据生成逻辑

$$r_i = \bar{r} + \gamma \cdot (\text{BM}_i - \overline{\text{BM}}) + \varepsilon_i$$

其中：
- $\bar{r}$ = 基础收益率（所有股票共享）
- $\gamma$ = 价值效应系数（> 0 表示高 B/M 带来高收益）
- $\varepsilon_i$ = 个股噪声

In [ ]:
# ========== 构造 10 只股票的微型数据 ==========
np.random.seed(42)

stocks = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10']

# B/M（账面市值比）：从低到高
bm_ratio = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0, 1.3, 1.8])

# 数据生成参数
base_return = 1.0    # 月基础收益率 1%
gamma = 1.5          # 价值效应：每 0.1 B/M → 0.15% 额外收益
noise = np.random.normal(0, 2.0, 10)

# 收益率 = 基础 + 价值效应 + 噪声
returns = base_return + gamma * (bm_ratio - bm_ratio.mean()) + noise

# 构建 DataFrame
df_micro = pd.DataFrame({
    'Stock': stocks,
    'B/M': bm_ratio,
    'Return (%)': np.round(returns, 2)
})

print("📋 10 只股票数据集：")
print(df_micro.to_string(index=False))
print(f"\n💡 价值效应系数 γ = {gamma}（每 0.1 B/M → 本期多赚 {gamma*0.1:.2f}%）")
print(f"   基础收益率 = {base_return}%，噪声标准差 = 2.0%")

### 📐 步骤 1: 按 B/M 排序分组

将 10 只股票按 B/M 从低到高排序，分为 2 组：
- **Low 组（Q1）**：B/M 最低的 5 只（成长股）→ 做空
- **High 组（Q2）**：B/M 最高的 5 只（价值股）→ 做多

$$\text{HML} = \bar{R}_{\text{High}} - \bar{R}_{\text{Low}}$$

In [ ]:
# ========== 步骤 1: 按 B/M 排序分组 ==========
print("📊 步骤 1: 按 B/M 排序分组")
print("─" * 55)

df_sorted = df_micro.sort_values('B/M').reset_index(drop=True)
df_sorted['Group'] = ['Low'] * 5 + ['High'] * 5

print("\n  Low B/M 组（成长股，做空）:")
for _, row in df_sorted[df_sorted['Group'] == 'Low'].iterrows():
    print(f"    {row['Stock']}: B/M = {row['B/M']:.1f},  收益 = {row['Return (%)']:>6.2f}%")

print("\n  High B/M 组（价值股，做多）:")
for _, row in df_sorted[df_sorted['Group'] == 'High'].iterrows():
    print(f"    {row['Stock']}: B/M = {row['B/M']:.1f},  收益 = {row['Return (%)']:>6.2f}%")

In [ ]:
# ========== 步骤 2: 计算各组平均收益率和 Spread ==========
print("📊 步骤 2: 计算各组平均收益率和 Spread")
print("─" * 55)

low_mean = df_sorted[df_sorted['Group'] == 'Low']['Return (%)'].mean()
high_mean = df_sorted[df_sorted['Group'] == 'High']['Return (%)'].mean()
spread = high_mean - low_mean

print(f"\n  Low B/M 组平均收益率  = {low_mean:.2f}%")
print(f"  High B/M 组平均收益率 = {high_mean:.2f}%")
print(f"\n  📐 HML = High - Low = {high_mean:.2f}% - {low_mean:.2f}% = {spread:.2f}%")
print(f"\n  💡 解读：价值股比成长股{'多' if spread > 0 else '少'}赚 {abs(spread):.2f}%")

---

## 3. 扩展到完整模拟：300 只股票 × 60 月

### 📐 数据生成过程 (DGP)

每月生成 300 只股票的截面数据：

$$r_{i,t} = \bar{r}_t + \gamma \cdot (\text{BM}_{i,t} - \overline{\text{BM}}_t) + \varepsilon_{i,t}$$

其中：
- $\bar{r}_t$ = 每月基础收益率（从 $N(1.0, 0.5)$ 抽样）
- $\gamma$ = 价值效应强度
- $\varepsilon_{i,t} \sim N(0, 3)$ = 个股噪声

In [ ]:
# ========== 完整模拟参数 ==========
N_STOCKS = 300    # 每月 300 只股票
N_MONTHS = 60     # 60 个月
N_GROUPS = 5      # 分 5 组

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"  分组数量: {N_GROUPS} 组")

In [ ]:
# ========== 生成模拟数据 ==========
np.random.seed(42)

all_data = []
for t in range(N_MONTHS):
    # B/M：对数正态分布（模拟 A 股 B/M 分布）
    bm = np.random.lognormal(mean=-0.5, sigma=0.6, size=N_STOCKS)
    bm = np.clip(bm, 0.05, 5.0)  # 限制范围
    
    # 基础收益率（每月不同）
    base_return = np.random.normal(1.0, 0.5)
    
    # 价值效应：高 B/M 收益更高
    gamma = 0.8
    value_effect = gamma * (bm - bm.mean())
    
    # 噪声
    noise = np.random.normal(0, 3.0, N_STOCKS)
    
    # 收益率
    returns = base_return + value_effect + noise
    
    month_df = pd.DataFrame({
        'Month': t + 1,
        'Stock': [f'S{i:03d}' for i in range(N_STOCKS)],
        'BM': bm,
        'Return': returns
    })
    all_data.append(month_df)

df_all = pd.concat(all_data, ignore_index=True)

print(f"✅ 数据生成完成：{len(df_all)} 条记录")
print(f"   股票数: {df_all['Stock'].nunique()}")
print(f"   月份数: {df_all['Month'].nunique()}")
print(f"\n📊 B/M 分布统计：")
print(df_all['BM'].describe().round(4))

### 📐 步骤 3: 每月按 B/M 分组

每月末将股票按 B/M 从低到高分成 5 组：
- Q1 = B/M 最低（成长股）
- Q5 = B/M 最高（价值股）

$$\text{HML}_t = \bar{R}_{Q5,t} - \bar{R}_{Q1,t}$$

In [ ]:
# ========== 每月按 B/M 分组 ==========
def assign_value_group(month_df):
    """按 B/M 从低到高分 5 组"""
    month_df = month_df.copy()
    month_df['ValueGroup'] = pd.qcut(month_df['BM'], N_GROUPS, 
                                      labels=['Q1(Low)', 'Q2', 'Q3', 'Q4', 'Q5(High)'])
    return month_df

df_all = df_all.groupby('Month', group_keys=False).apply(assign_value_group)

# 计算每月各组平均收益率
monthly_group_returns = df_all.groupby(['Month', 'ValueGroup'])['Return'].mean().unstack()

# 计算每月 HML = Q5 - Q1
monthly_spreads = monthly_group_returns['Q5(High)'] - monthly_group_returns['Q1(Low)']

print("📊 每月各组平均收益率（前 5 个月）：")
print(monthly_group_returns.head().round(3))
print(f"\n📊 每月 HML Spread（前 5 个月）：")
print(monthly_spreads.head().round(3))

### 📐 步骤 4: T 检验——HML 是否显著大于零？

$$t = \frac{\overline{\text{HML}}}{s_{\text{HML}} / \sqrt{T}}$$

其中：
- $\overline{\text{HML}}$ = HML 时间序列均值
- $s_{\text{HML}}$ = HML 样本标准差
- $T$ = 月份数

In [ ]:
# ========== T 检验 ==========
spreads = monthly_spreads.values
spread_mean = spreads.mean()
spread_std = spreads.std(ddof=1)
spread_se = spread_std / np.sqrt(N_MONTHS)
t_stat = spread_mean / spread_se
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=N_MONTHS - 1))

print("📊 步骤 4: T 检验结果")
print("─" * 55)
print(f"  HML 均值       = {spread_mean:.4f}%")
print(f"  HML 标准差     = {spread_std:.4f}%")
print(f"  标准误         = {spread_se:.4f}%")
print(f"  T 统计量       = {t_stat:.4f}")
print(f"  P 值 (双尾)    = {p_value:.6f}")
print(f"\n  💡 解读：")
if abs(t_stat) > 2.6:
    print(f"  ✓ |t| = {abs(t_stat):.2f} > 2.6 → 在 1% 水平下显著")
elif abs(t_stat) > 2.0:
    print(f"  ✓ |t| = {abs(t_stat):.2f} > 2.0 → 在 5% 水平下显著")
else:
    print(f"  ✗ |t| = {abs(t_stat):.2f} < 2.0 → 不显著")

In [ ]:
# ========== 用 scipy 验证 ==========
t_scipy, p_scipy = stats.ttest_1samp(spreads, 0)

print("🔬 scipy.stats.ttest_1samp 验证:")
print(f"  手算 T 统计量 = {t_stat:.6f}")
print(f"  scipy T 统计量 = {t_scipy:.6f}")
print(f"  手算 P 值     = {p_value:.6f}")
print(f"  scipy P 值    = {p_scipy:.6f}")
print(f"\n  ✅ 验证通过！")

### 📐 步骤 5: 经济意义——夏普比率

$$\text{SR}_{\text{monthly}} = \frac{\overline{\text{HML}}}{s_{\text{HML}}}$$

$$\text{SR}_{\text{annual}} = \text{SR}_{\text{monthly}} \times \sqrt{12}$$

In [ ]:
# ========== 夏普比率 ==========
sr_monthly = spread_mean / spread_std
sr_annual = sr_monthly * np.sqrt(12)

print("📊 步骤 5: 夏普比率")
print("─" * 55)
print(f"  月度夏普比率 = {sr_monthly:.4f}")
print(f"  年化夏普比率 = {sr_annual:.4f}")
print(f"\n  📐 验证: t = SR_monthly × √T = {sr_monthly:.4f} × √{N_MONTHS} = {sr_monthly * np.sqrt(N_MONTHS):.4f}")
print(f"\n  💡 解读：年化 SR > 0.5 表示因子有较好的风险调整后收益")

---

## 4. 单调性检验：各组收益率是否单调递增？

### 📐 Spearman 秩相关

检验 B/M 分组（Q1 到 Q5）与平均收益率之间是否存在单调关系：

$$\rho_s = 1 - \frac{6 \sum d_i^2}{n(n^2 - 1)}$$

- $|\rho_s| > 0.8$：强单调性
- $|\rho_s| > 0.5$：中等单调性
- $|\rho_s| < 0.5$：弱单调性

In [ ]:
# ========== 单调性检验 ==========
avg_group_returns = monthly_group_returns.mean()
group_ranks = np.arange(1, N_GROUPS + 1)
group_ret_values = avg_group_returns.values

sp_corr, sp_p = stats.spearmanr(group_ranks, group_ret_values)

print("📊 单调性检验结果")
print("─" * 55)
print(f"  各组平均收益率：")
for i, (group, ret) in enumerate(avg_group_returns.items()):
    print(f"    {group}: {ret:.4f}%")
print(f"\n  Spearman 秩相关系数 = {sp_corr:.4f}")
print(f"  P 值 = {sp_p:.6f}")
print(f"\n  💡 解读：")
if abs(sp_corr) > 0.8:
    print(f"  ✓ |ρ| = {abs(sp_corr):.2f} > 0.8 → 强单调性，因子质量优秀")
elif abs(sp_corr) > 0.5:
    print(f"  ✓ |ρ| = {abs(sp_corr):.2f} > 0.5 → 中等单调性")
else:
    print(f"  ✗ |ρ| = {abs(sp_corr):.2f} < 0.5 → 弱单调性，因子不可靠")

---

## 5. 可视化：价值效应的全景图

In [ ]:
# ========== 可视化 T 检验 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 图1: Spread 时间序列 ---
ax1 = axes[0]
months = range(1, N_MONTHS + 1)
ax1.bar(months, spreads, color=['#2ecc71' if s > 0 else '#e74c3c' for s in spreads],
        alpha=0.7, edgecolor='none')
ax1.axhline(y=spread_mean, color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {spread_mean:.2f}%')
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Month', fontsize=11)
ax1.set_ylabel('HML Spread (%)', fontsize=11)
ax1.set_title('Monthly HML Spread (High BM - Low BM)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- 图2: Spread 分布 ---
ax2 = axes[1]
ax2.hist(spreads, bins=15, edgecolor='black', alpha=0.7, color='steelblue', density=True)
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='$H_0$: $\mu$=0')
ax2.axvline(x=spread_mean, color='blue', linestyle='--', linewidth=2,
            label=f'Sample Mean = {spread_mean:.2f}%')
x_fit = np.linspace(spreads.min() - 1, spreads.max() + 1, 100)
ax2.plot(x_fit, stats.norm.pdf(x_fit, spread_mean, spread_std), 'b-', linewidth=2)
ax2.set_xlabel('HML Spread (%)', fontsize=11)
ax2.set_ylabel('Probability Density', fontsize=11)
ax2.set_title('HML Spread Distribution', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

# --- 图3: T 分布与 T 值位置 ---
ax3 = axes[2]
x_t = np.linspace(-5, 5, 1000)
t_dist = stats.t.pdf(x_t, df=N_MONTHS - 1)
ax3.plot(x_t, t_dist, 'b-', linewidth=2, label=f'T distribution (df={N_MONTHS-1})')
ax3.fill_between(x_t, t_dist, where=(x_t >= abs(t_stat)), color='red', alpha=0.4)
ax3.fill_between(x_t, t_dist, where=(x_t <= -abs(t_stat)), color='red', alpha=0.4,
                 label=f'P-value region = {p_value:.4f}')
ax3.axvline(x=t_stat, color='red', linestyle='--', linewidth=2,
            label=f't = {t_stat:.2f}')
ax3.set_xlabel('T Value', fontsize=11)
ax3.set_ylabel('Probability Density', fontsize=11)
ax3.set_title('T Distribution and T Value Position', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：每月 HML Spread 的时间序列，绿色为正（价值股跑赢），红色为负")
print(f"  图2：HML Spread 的分布直方图，叠加正态拟合曲线")
print(f"  图3：T 分布图，红色区域为拒绝域，t 值落在该区域表示显著")

In [ ]:
# ========== 可视化单调性 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 图1: 各组收益柱状图 ---
ax1 = axes[0]
group_labels = avg_group_returns.index.tolist()
group_vals = avg_group_returns.values
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, N_GROUPS))
bars = ax1.bar(group_labels, group_vals, color=colors, edgecolor='black', alpha=0.85)
for bar, v in zip(bars, group_vals):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
             f'{v:.3f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax1.plot(group_labels, group_vals, 'ko--', linewidth=2, markersize=8, zorder=5)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Value Group (Q1=Low BM → Q5=High BM)', fontsize=11)
ax1.set_ylabel('Average Monthly Return (%)', fontsize=11)
ax1.set_title(f'Value Monotonicity (ρ = {sp_corr:.3f})', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# --- 图2: 累计收益率 ---
ax2 = axes[1]
cum_hml = np.cumsum(spreads)
ax2.plot(range(1, N_MONTHS + 1), cum_hml, 'b-', linewidth=2)
ax2.fill_between(range(1, N_MONTHS + 1), cum_hml, alpha=0.3, color='steelblue')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Month', fontsize=11)
ax2.set_ylabel('Cumulative HML Return (%)', fontsize=11)
ax2.set_title('Cumulative HML Factor Return', fontsize=13, fontweight='bold')
ax2.grid(alpha=0.3)

# --- 图3: 多空两头累计收益 ---
ax3 = axes[2]
cum_high = np.cumsum(monthly_group_returns['Q5(High)'].values)
cum_low = np.cumsum(monthly_group_returns['Q1(Low)'].values)
ax3.plot(range(1, N_MONTHS + 1), cum_high, 'g-', linewidth=2, label='High BM (Value)')
ax3.plot(range(1, N_MONTHS + 1), cum_low, 'r-', linewidth=2, label='Low BM (Growth)')
ax3.axhline(y=0, color='black', linewidth=0.8)
ax3.set_xlabel('Month', fontsize=11)
ax3.set_ylabel('Cumulative Return (%)', fontsize=11)
ax3.set_title('Long and Short Leg Cumulative Returns', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图1：各 B/M 组的平均月收益率，理想情况下应单调递增（高 B/M 收益最高）")
print(f"  图2：HML 因子的累计收益率曲线，上升趋势表示价值股持续跑赢")
print(f"  图3：多空两头各自的累计收益，绿色=高 B/M（做多），红色=低 B/M（做空）")

---

## 6. Fama-MacBeth 回归：控制市值后的价值效应

### 📐 Fama-MacBeth 两步回归

第一步：每月截面回归
$$r_{i,t} = \alpha_t + \beta_t \cdot \text{BM}_{i,t} + \varepsilon_{i,t}$$

第二步：对 $\hat{\beta}_t$ 时间序列取均值和标准误
$$\bar{\beta} = \frac{1}{T} \sum_{t=1}^T \hat{\beta}_t$$

In [ ]:
# ========== Fama-MacBeth 回归 ==========
betas = []
for month in range(1, N_MONTHS + 1):
    month_data = df_all[df_all['Month'] == month].copy()
    X = sm.add_constant(month_data['BM'])
    y = month_data['Return']
    model = sm.OLS(y, X).fit()
    betas.append(model.params['BM'])

betas = np.array(betas)
beta_mean = betas.mean()
beta_std = betas.std(ddof=1)
beta_se = beta_std / np.sqrt(N_MONTHS)
t_beta = beta_mean / beta_se
p_beta = 2 * (1 - stats.t.cdf(abs(t_beta), df=N_MONTHS - 1))

print("📊 Fama-MacBeth 回归结果")
print("─" * 55)
print(f"  BM 系数均值   = {beta_mean:.4f}")
print(f"  BM 系数标准误 = {beta_se:.4f}")
print(f"  T 统计量      = {t_beta:.4f}")
print(f"  P 值           = {p_beta:.6f}")
print(f"\n  💡 解读：")
if abs(t_beta) > 2.0:
    print(f"  ✓ BM 系数显著（t={t_beta:.2f}），控制其他因素后价值效应依然存在")
else:
    print(f"  ✗ BM 系数不显著（t={t_beta:.2f}），价值效应可能是虚假的")

---

## 7. 结果汇总报告

In [ ]:
# ========== 汇总报告 ==========
print("=" * 60)
print("📋 价值因子 (HML) 实证分析报告")
print("=" * 60)

print(f"\n🎯 研究问题:")
print(f"   高 B/M（价值股）是否比低 B/M（成长股）获得更高收益？")

print(f"\n📊 数据概况:")
print(f"   股票数量: {N_STOCKS} 只/月")
print(f"   时间跨度: {N_MONTHS} 个月")
print(f"   分组方式: 按 B/M 从低到高分 {N_GROUPS} 组")

print(f"\n🧮 统计检验:")
print(f"   HML 月均收益率 = {spread_mean:.4f}%")
print(f"   T 统计量      = {t_stat:.4f}")
print(f"   P 值           = {p_value:.6f}")
print(f"   月度夏普比率   = {sr_monthly:.4f}")
print(f"   年化夏普比率   = {sr_annual:.4f}")
print(f"   Spearman ρ    = {sp_corr:.4f}")

print(f"\n🧮 Fama-MacBeth 回归:")
print(f"   BM 系数        = {beta_mean:.4f}")
print(f"   T 统计量      = {t_beta:.4f}")

print(f"\n🎯 结论:")
if t_stat > 2.0 and sp_corr > 0.5:
    print(f"  ✓ 价值效应显著：高 B/M 股票收益显著高于低 B/M 股票")
    print(f"  ✓ 单调性良好：收益率随 B/M 增加而单调递增")
elif t_stat > 2.0:
    print(f"  ✓ 价值效应显著，但单调性不佳")
else:
    print(f"  ✗ 价值效应不显著")

print("\n" + "=" * 60)

---

## 📌 核心概念回顾

### 📌 价值因子 (HML)
- **定义**: High-Minus-Low，做多高 B/M（价值股）、做空低 B/M（成长股）的多空组合
- **公式**: $\text{HML} = R_{\text{High BM}} - R_{\text{Low BM}}$
- **含义**: 反映价值股相对成长股的超额收益
- **判断标准**: t > 2.0 表示显著，Spearman ρ > 0.8 表示强单调性

### 📌 B/M 的计算
- **公式**: B/M = 归股东权益合计 / 总市值
- **滞后处理**: 使用上一年年报数据，避免前视偏差
- **A 股特殊性**: EP 可能优于 BM（Liu et al. 2019）

### 📌 Fama-MacBeth 回归
- **目的**: 控制其他因素后检验变量的增量解释力
- **步骤**: 每月截面回归 → 系数时间序列 → T 检验
- **优点**: 可以同时控制多个变量

### 🔗 完整流程
```
B/M 数据 → 每月排序分组 → 计算各组收益率 → HML = High - Low
    ↓           ↓              ↓              ↓
  滞后处理     5 组          等权/市值加权      T 检验 + 单调性
```

---

## ❌ 常见误区

### ❌ 误区 1: B/M 越高越好
**✓ 正确理解**: 高 B/M 可能反映财务困境（"价值陷阱"）。需要结合盈利、现金流等指标综合判断。

### ❌ 误区 2: 价值因子在所有时期都有效
**✓ 正确理解**: 价值因子有周期性。2008 年后全球价值因子表现不佳，A 股也有类似现象。

### ❌ 误区 3: 只用当期 B/M 就够了
**✓ 正确理解**: 必须使用滞后数据（上一年年报），否则会有前视偏差（look-ahead bias）。

### ❌ 误区 4: BM 和 EP 是完全等价的
**✓ 正确理解**: EP = BM × ROE。在 A 股中，由于小市值和盈利因子的交互，EP 可能优于 BM。

### ❌ 误区 5: 价值因子的收益是"免费午餐"
**✓ 正确理解**: 价值因子收益是对承担风险的补偿。高 B/M 企业面临更大的财务困境风险和经营杠杆风险。